# CognitivRec ? Group E

Run the setup cell, then the four core stages or the rolling time-cutoff study. The cells call the same Python modules as the command line.

**Local setup.** Run uv sync --locked from CognitivRec and select the project's Python environment as the kernel.

**Colab setup.** Select a CPU runtime. Place the approved repository copy under /content/CognitivRec, including the src and configs directories. If the runtime lacks dependencies, run %pip install -e /content/CognitivRec in a setup cell and restart the kernel before running this notebook.

MovieLens data is loaded from data/ml-100k. If the download is blocked, place u.data and u.item from the official archive in that directory before running the experiments.

In [ ]:
# Keep QUICK=True for a smoke run. Set it to False for report-quality results.
# Keep RUN_CORE=False if you only need the rolling follow-up and the demo below.
QUICK = True
RUN_CORE = True
RUN_TEMPORAL_FOLLOWUP = True

import os
import sys
from datetime import datetime, timezone
from pathlib import Path

# Set these before loading NumPy or Matplotlib in a fresh kernel.
for name in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[name] = "1"

cwd = Path.cwd().resolve()
candidates = (cwd, cwd / "CognitivRec", *cwd.parents)
root = next((p for p in candidates if (p / "configs/temporal_followup.yaml").is_file()), None)
if root is None:
    raise FileNotFoundError(
        "Open the notebook in the current CognitivRec repository or prepare "
        "/content/CognitivRec as described above."
    )

os.environ.setdefault("MPLCONFIGDIR", str(root / "results" / ".matplotlib"))
sys.path.insert(0, str(root / "src"))

from IPython.display import Image, display
from cognitiverec.config import load_config
from cognitiverec.experiments import (
    exploration_study,
    paper_figures,
    stage1_baselines,
    stage2_cognitive,
    stage3_trust,
    stage4_ablation,
    temporal_followup,
)

# Give this notebook session its own results and tuning files.
mode = "smoke" if QUICK else "full"
stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%fZ")
output = root / "results" / f"notebook-{mode}" / stamp
overrides = {"paths": {"results": str(output), "tuned": str(output / "tuned")}}
if QUICK:
    overrides["seeds"] = [42]

print(f"{mode.title()} run: {output}")

## 1. Build the split and tune the original baselines

Jobin and Mudit. Review the baseline metrics and the cohort counts before interpreting test performance.

In [ ]:
# Tune Popularity, ItemKNN, and SVD using ratings prior to the test window.
if RUN_CORE:
    config = load_config("stage1_baselines", overrides)
    if QUICK:
        config["baselines"]["ItemKNN"] = {
            "top_k": [50, 150], "shrinkage": [0.0],
            "binary": [False], "tfidf": [True],
        }
    stage1 = stage1_baselines.run(config)
    display(stage1.tables["test"][["model", "ndcg", "recall", "coverage_pct"]])

## 2. Tune Models C and D

Mudit and Sai Abhinav. C is selected by validation replay. D selects the highest-coverage bonus that stays within the validation NDCG budget.

In [ ]:
# Write the selected model parameters into this session's tuning directory.
if RUN_CORE:
    config = load_config("stage2_cognitive", overrides)
    if QUICK:
        config["search"]["n_trials"] = 3
    stage2 = stage2_cognitive.run(config)
    display(stage2.tables["beta"])

## 3. Evaluate the four research questions

All three members. The test run reads the tuning files written by Stages 1 and 2 in this session.

In [ ]:
# Inspect the CSVs for the full fairness, fidelity, and reliability metrics.
if RUN_CORE:
    stage3 = stage3_trust.run(load_config("stage3_trust", overrides))
    for figure in stage3.figures:
        display(Image(filename=str(figure), width=800))

## 4. Isolate mechanisms and inspect trade-offs

Mudit and Sai Abhinav. Compare the matched ablations and read the tuning curves on validation.

In [ ]:
# Check the change from full D when each component is disabled.
if RUN_CORE:
    stage4 = stage4_ablation.run(load_config("stage4_ablation", overrides))
    for figure in stage4.figures:
        display(Image(filename=str(figure), width=660))

## 5. Test sensitivity to the calendar cutoff

Mudit runs the rolling windows, Jobin checks ranking and exposure, and Sai Abhinav verifies the demo cases. All eight models use the same candidate mask within each window. EASE and the matched all-history and recency profiles provide the added comparisons.

In [ ]:
# This study retunes within each window and can run independently of Stages 1?4.
if RUN_TEMPORAL_FOLLOWUP:
    followup_config = temporal_followup.study_config(quick=QUICK)
    followup_config["paths"]["results"] = str(output)
    followup = temporal_followup.run(followup_config)
    display(followup.tables["windows"][[
        "window", "test_first_utc", "test_last_utc", "returning_users", "ranking_users",
    ]])
    display(followup.tables["metrics"][[
        "window", "model", "ndcg", "replay_ndcg", "coverage_pct",
        "rank_merit_gap", "recall_tail",
    ]])
    print(f"Manifest and CSVs: {followup.run_dir}")

## 6. Compare the exploration policy across scorers

Use the frozen EASE and Cognitive parameters from the rolling run. Tune the policy separately for each scorer with the same validation loss budget. Compare four variants on the test windows and export the paper figures.

In [ ]:
# Run this cell after the rolling follow-up. Its source run determines the rating windows.
if RUN_TEMPORAL_FOLLOWUP:
    policy = exploration_study.run(followup.run_dir, quick=QUICK)
    display(policy.tables["effects"].round(4))
    # Use SVG for vector placement and PNG when inserting a raster image.
    figures = paper_figures.render(
        policy.run_dir, followup.run_dir, policy.run_dir / "figures",
        trust=stage3.run_dir if RUN_CORE else None,
        ablation=stage4.run_dir if RUN_CORE else None,
    )
    for figure in figures:
        print(f"{figure.name} and {figure.with_suffix('.svg').name}")
    display(Image(filename=str(figures[1]), width=840))

## 7. Demonstrate explanation fidelity

Tell the audience the original recommendation and its highest-attended memory. Compare the score of the original recommendation before and after deleting that memory. Then reveal the post-deletion top recommendation. The examples show how the same explanation rule can identify influential evidence and can also highlight an entry whose removal increases the score.

In [ ]:
# Use the deterministic case selection from the fitted Model D in the last window.
# Save the run folder if you want to recover the exact same demo later.
if RUN_TEMPORAL_FOLLOWUP:
    cases = followup.tables["demo_examples"]
    if cases.empty:
        print("This run produced no eligible deletion cases.")
    else:
        display(cases[[
            "selection", "recommended_title", "evidence_title", "base_score",
            "score_after_deletion", "top_after_deletion_title",
        ]].round(4))